# DisputeCourt — GRPO on free Colab (T4)

Runtime > Change runtime type > **T4 GPU**, then Runtime > Run all.

This notebook deliberately does **not** install `trl`. GRPO is implemented
directly in `training/grpo_minimal.py` against plain torch + transformers +
peft, because the TRL/torchao/peft version matrix on Colab breaks often and
debugging it is not a good use of a deadline.

End to end: ~25-35 min on a free T4.


## 1. Check the GPU


In [ ]:
!nvidia-smi
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Set Runtime > Change runtime type > T4 GPU'


## 2. Install

Only `peft`. torch and transformers already ship with Colab, and *not*
touching them is what keeps this reproducible.


In [ ]:
# Colab ships torchao 0.10.0. peft's LoRA dispatcher calls
# is_torchao_available(), which RAISES (rather than returning False) on an
# incompatible torchao version, so get_peft_model() dies before it ever
# reaches our layers. We never want the torchao path -- removing the package
# makes that check return False and peft skips it.
!pip uninstall -y -q torchao
!pip install -q 'peft>=0.11.0'

import peft, transformers, torch
print('peft', peft.__version__, '| transformers', transformers.__version__,
      '| torch', torch.__version__)

# Fail here, loudly, rather than 200 GRPO steps later.
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM
_m = AutoModelForCausalLM.from_pretrained('hf-internal-testing/tiny-random-gpt2')
_m = get_peft_model(_m, LoraConfig(task_type='CAUSAL_LM', target_modules=['c_attn']))
print('LoRA injection works')
del _m


## 3. Get the code


In [ ]:
!git clone -q https://github.com/AniketAslaliya/disputecourt.git /content/disputecourt || (cd /content/disputecourt && git pull -q)
%cd /content/disputecourt
!ls


## 4. Baseline: the base model, before any RL

Same prompt, same eval split, same parser as the tuned run below. The only
difference between the two is the LoRA adapter, which is what makes the
delta attributable to GRPO rather than to harness changes.


In [ ]:
!python eval/run_model_eval.py \
    --model Qwen/Qwen2.5-0.5B-Instruct \
    --out data/results_base.jsonl \
    --label 'Base Qwen2.5-0.5B (no RL)'


## 5. GRPO

`--steps` is the number of prompts (one group of `--group-size` completions
each). 250 x 6 is roughly 20 min on a T4. Watch `reward(last20)` in the log:
if it climbs, the policy is learning; if it flatlines while
`skipped (zero-variance groups)` grows, the policy has collapsed onto one
verdict and the run is telling you so.


In [ ]:
!python training/grpo_minimal.py \
    --steps 250 \
    --group-size 6 \
    --max-new-tokens 128 \
    --lr 1e-5 \
    --kl-beta 0.02 \
    --accum 2 \
    --output training/checkpoints


## 6. Evaluate the tuned policy


In [ ]:
!python eval/run_model_eval.py \
    --model Qwen/Qwen2.5-0.5B-Instruct \
    --adapter training/checkpoints \
    --out data/results_grpo.jsonl \
    --label 'GRPO-tuned'


## 7. The table

Keyword control vs base model vs GRPO-tuned, on the same 100 held-out cases.


In [ ]:
!python eval/keyword_baseline.py
!python eval/compare_all.py


## 8. Training curve


In [ ]:
import json, matplotlib.pyplot as plt
h = json.load(open('training/checkpoints/train_history.json'))
scored = [x for x in h if not x['skipped']]
r = [x['reward_mean'] for x in scored]
w = 20
smooth = [sum(r[max(0,i-w):i+1])/len(r[max(0,i-w):i+1]) for i in range(len(r))]
plt.figure(figsize=(9,4))
plt.plot(r, alpha=0.25, label='per-group mean reward')
plt.plot(smooth, lw=2, label=f'{w}-group moving average')
plt.xlabel('GRPO step'); plt.ylabel('reward'); plt.legend()
plt.title('DisputeCourt GRPO training reward'); plt.grid(alpha=0.3)
plt.tight_layout(); plt.savefig('data/training_curve.png', dpi=140)
plt.show()
print('skipped zero-variance groups:', sum(1 for x in h if x['skipped']), '/', len(h))


## 9. Download the results

Pull these four files down and commit them to the repo -- they are the
evidence behind the README's metrics table.


In [ ]:
from google.colab import files
for f in ['data/results_base.jsonl', 'data/results_grpo.jsonl',
          'data/results_base.summary.json', 'data/results_grpo.summary.json',
          'data/comparison.json', 'data/training_curve.png']:
    try:
        files.download(f)
    except Exception as e:
        print('skip', f, e)
